# 🧠 MBTI Personality Classification — RAG + Multi-Agent (Local LLM)

**Constraints:** No fine-tuning · No paid API · Local LLM · Colab/Kaggle T4

**Architecture:** 6 Agents — Feature Extraction → Dual-RAG → 4×Analysis → Synthesis

**LLM:** Qwen2.5-7B-Instruct @ 4-bit quantization (~4.5 GB VRAM)

Run cells top-to-bottom. Set `KB_SAMPLE` and `TEST_SAMPLE` to small numbers for fast testing.

## 📦 Cell 1 — Install (FIXED v3 — numpy pin + restart)

In [1]:
# ============================================================
# CELL 1 — Cài đặt môi trường (FIXED v3)
#
# Root cause của lỗi numpy.dtype size changed:
#   Kaggle pre-installs numpy 2.x, nhưng tensorflow/jax trên
#   image này được build với numpy 1.x ABI.
#   Khi pip install ghi đè numpy → binary incompatibility.
#
# Fix: Pin numpy==1.26.4 TRƯỚC, cài xong PHẢI restart kernel.
# ============================================================

import subprocess, sys

# ── Bước 1: Uninstall numpy hiện tại, pin lại đúng version ──
print("⏳ Pinning numpy to 1.26.4 ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy==1.26.4", "--force-reinstall"], check=True)

# ── Bước 2: Cài các thư viện còn lại ────────────────────────
pkgs = [
    "langchain==0.2.16",
    "langchain-community==0.2.16",
    "langchain-huggingface==0.0.3",
    "faiss-cpu",                   # faiss-gpu không có trên PyPI
    "sentence-transformers==3.0.1",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.33.0",
    "transformers==4.44.2",
    "einops",
    "scipy",
    "scikit-learn",
    "pandas",
    "tqdm",
]
print("⏳ Installing packages ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ All packages installed")

# ── Bước 3: BẮTBUỘC restart kernel ───────────────────────────
# Numpy binary phải được load lại từ đầu sau khi reinstall.
# Nếu không restart, lỗi numpy.dtype sẽ vẫn còn.
print("\n" + "="*55)
print("⚠️  KERNEL RESTART REQUIRED")
print("   Sau khi cell này chạy xong, hãy:")
print("   Kaggle : Session → Restart (không Run All)")
print("   Colab  : Runtime → Restart session")
print("   Sau đó SKIP cell này, chạy từ Cell 2 trở đi.")
print("="*55)

# Auto-restart (hoạt động trên Colab, Kaggle thì manual)
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
except Exception:
    print("   (Auto-restart không thành công → restart thủ công)")

⏳ Pinning numpy to 1.26.4 ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

⏳ Installing packages ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.3 MB/s eta 0:00:00
✅ All packages installed


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
google-adk 1.25.1 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.2.43 which is incompatible.


## 🤖 Cell 2 — LLM Backend (FIXED v3 — TF/JAX isolation)

In [2]:
# ============================================================
# CELL 2 — Load Local LLM (FIXED v3)
#
# Root cause chi tiết:
#   transformers.pipelines import → image_transforms.py
#   → if is_tf_available(): import tensorflow as tf
#   → tensorflow import jax → jax import numpy C extensions
#   → numpy ABI mismatch → RuntimeError
#
# Fix strategy:
#   1. Ngăn transformers auto-import TF/JAX bằng env vars
#   2. Import AutoModel TRƯỚC khi bất kỳ thứ gì load tf
#   3. Dùng pipeline với trust_remote_code explicit
# ============================================================

import os

# ── QUAN TRỌNG: Tắt TF/JAX trước khi import transformers ────
# Đây là nguyên nhân gốc rễ của lỗi numpy ABI
os.environ["USE_TF"]                  = "0"
os.environ["USE_JAX"]                 = "0"
os.environ["TRANSFORMERS_NO_TF"]      = "1"  # tắt TF backend
os.environ["TF_CPP_MIN_LOG_LEVEL"]    = "3"  # suppress TF logs
os.environ["TOKENIZERS_PARALLELISM"]  = "false"

import sys
import torch

# Verify numpy version trước khi import transformers
import numpy as np
print(f"✅ numpy version: {np.__version__}  (cần 1.x)")
if np.__version__.startswith("2."):
    raise RuntimeError(
        "❌ numpy 2.x detected! Chạy lại Cell 1 rồi RESTART KERNEL.\n"
        "   Kaggle: Session → Restart | Colab: Runtime → Restart session"
    )

# ── Import transformers SAU khi đã set env vars ──────────────
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
# Import pipeline riêng với explicit backend override
from transformers import pipeline as hf_pipeline_fn

print("✅ transformers imported OK (no TF/JAX conflict)")

# ── Import LangChain wrapper (cascade fallback) ───────────────
HuggingFacePipeline = None
for import_path in [
    ("langchain_huggingface", "HuggingFacePipeline"),
    ("langchain_community.llms", "HuggingFacePipeline"),
    ("langchain.llms", "HuggingFacePipeline"),
]:
    try:
        mod = __import__(import_path[0], fromlist=[import_path[1]])
        HuggingFacePipeline = getattr(mod, import_path[1])
        print(f"✅ HuggingFacePipeline from: {import_path[0]}")
        break
    except (ImportError, AttributeError):
        continue

if HuggingFacePipeline is None:
    raise ImportError("❌ HuggingFacePipeline not found — chạy lại Cell 1")

# ── GPU config ───────────────────────────────────────────────
NUM_GPUS = torch.cuda.device_count()
print(f"\n🖥  GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"   GPU {i}: {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)")

# ── Model selection ──────────────────────────────────────────
# Qwen2.5-7B: JSON output tốt, 4-bit ~4.5GB VRAM, no gating
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
# Fallback nếu download chậm:
# MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # ~2.5 GB VRAM

# ── 4-bit quantization config ────────────────────────────────
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # ~0.4 bpp thêm, tăng acc
)

# ── Tokenizer ────────────────────────────────────────────────
print(f"\n⏳ Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="left",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Model (4-bit, multi-GPU auto dispatch) ───────────────────
print("⏳ Loading model weights (4-bit NF4)...")
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",          # phân bổ tự động qua 2 GPU
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
llm_model.eval()

# ── HuggingFace pipeline ─────────────────────────────────────
hf_pipe = hf_pipeline_fn(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,         # [OOM] giới hạn output
    do_sample=False,            # greedy: nhanh, ít VRAM
    temperature=None,
    repetition_penalty=1.1,
    return_full_text=False,     # chỉ trả phần được sinh ra
)

# ── LangChain wrapper ────────────────────────────────────────
local_llm = HuggingFacePipeline(pipeline=hf_pipe)

# ── Smoke test ───────────────────────────────────────────────
print("\n⏳ Smoke test...")
test_out = local_llm.invoke('Reply with only the word: READY')
print(f"✅ LLM OK. Output: '{str(test_out).strip()[:50]}'")

torch.cuda.empty_cache()
for i in range(NUM_GPUS):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i} VRAM: {alloc:.2f} / {total:.2f} GB")

✅ numpy version: 2.0.2  (cần 1.x)


RuntimeError: ❌ numpy 2.x detected! Chạy lại Cell 1 rồi RESTART KERNEL.
   Kaggle: Session → Restart | Colab: Runtime → Restart session

## ⚙️ Cell 3 — Global Config & Constants

In [ ]:
# ============================================================
# CELL 3 — Config toàn cục & Constants
# ============================================================

import os, re
import numpy as np
import pandas as pd
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────
DATA_PATH  = "/kaggle/input/datasets/datasnaek/mbti-type/mbti_1.csv"   # thay bằng đường dẫn thực tế
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ── Preprocessing config ─────────────────────────────────────
MAX_POSTS = 50
MAX_WORDS = 70

# ── 16 MBTI types + từ liên quan cần mask ───────────────────
MBTI_16 = [
    "infj","infp","intj","intp",
    "isfj","isfp","istj","istp",
    "enfj","enfp","entj","entp",
    "esfj","esfp","estj","estp",
]
MBTI_EXTRA = [
    "introvert","extrovert","introverted","extroverted",
    "sensing","intuition","intuitive","thinking","feeling",
    "judging","perceiving","mbti","myers","briggs",
    "personality type","cognitive function",
]
ALL_MASK_WORDS = MBTI_16 + MBTI_EXTRA
MASK_REGEX = re.compile(
    r"\b(" + "|".join(re.escape(w) for w in ALL_MASK_WORDS) + r")\b",
    re.IGNORECASE,
)

# ── Label definitions ────────────────────────────────────────
# Trục → (tên class 0, tên class 1)
AXIS_LABELS = {
    "IE": ("I", "E"),   # 0=Introvert, 1=Extrovert
    "SN": ("S", "N"),   # 0=Sensing,   1=iNtuition
    "TF": ("T", "F"),   # 0=Thinking,  1=Feeling
    "JP": ("J", "P"),   # 0=Judging,   1=Perceiving
}
AXES       = list(AXIS_LABELS.keys())  # ["IE","SN","TF","JP"]
DIM_NAMES  = ["I/E", "S/N", "T/F", "J/P"]
LABEL_COLS = [f"label_{ax}" for ax in AXES]

# ── Label Enrichment (Knowledge Base cho Agents) ─────────────
# Mô tả đặc điểm ngôn ngữ của từng trục — dùng trong prompt
LABEL_ENRICHMENT = {
    "IE": (
        "I/E Axis — Introvert (0) vs Extrovert (1):\n"
        "Introverts: prefer solitary reflection, use 'I think/feel', "
        "write long introspective paragraphs, fewer exclamation marks, "
        "reference books/ideas/inner world.\n"
        "Extroverts: energized by social interaction, use 'we/us/let's', "
        "shorter sentences, more exclamation marks, reference events/people/activities."
    ),
    "SN": (
        "S/N Axis — Sensing (0) vs iNtuition (1):\n"
        "Sensing: concrete facts, specific details, past experience, "
        "practical examples, sequential descriptions, 'how' questions.\n"
        "iNtuition: abstract patterns, future possibilities, metaphors, "
        "theoretical frameworks, 'why' questions, big-picture thinking."
    ),
    "TF": (
        "T/F Axis — Thinking (0) vs Feeling (1):\n"
        "Thinking: logical analysis, impersonal criteria, "
        "pros/cons lists, objective language, 'what is correct'.\n"
        "Feeling: personal values, empathy language ('I feel', 'I care'), "
        "harmony-seeking, subjective experience, 'what is right'."
    ),
    "JP": (
        "J/P Axis — Judging (0) vs Perceiving (1):\n"
        "Judging: structured language, deadlines, plans, "
        "decisive statements, 'should/must', closure-seeking.\n"
        "Perceiving: open-ended language, spontaneous, "
        "'maybe/perhaps/let's see', flexible, process-oriented."
    ),
}

# ── Mapping MBTI string → 4 binary labels ────────────────────
def mbti_to_binary(mbti_str: str) -> dict:
    """'INTJ' → {'label_IE':0,'label_SN':1,'label_TF':0,'label_JP':0}"""
    m = mbti_str.strip().upper()
    return {
        "label_IE": 0 if m[0] == "I" else 1,
        "label_SN": 0 if m[1] == "S" else 1,
        "label_TF": 0 if m[2] == "T" else 1,
        "label_JP": 0 if m[3] == "J" else 1,
    }

def labels_to_mbti(ie: int, sn: int, tf: int, jp: int) -> str:
    """(0,1,0,0) → 'INTJ'"""
    return (
        ("I" if ie == 0 else "E") +
        ("S" if sn == 0 else "N") +
        ("T" if tf == 0 else "F") +
        ("J" if jp == 0 else "P")
    )

print("✅ Config & constants ready")
print(f"   AXES: {AXES}")
print(f"   Result dir: {os.path.abspath(RESULT_DIR)}")

## 🔧 Cell 4 — Load Data & Preprocessing

In [ ]:
# ============================================================
# CELL 4 — Load dữ liệu & Tiền xử lý
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# ── 4.1 Load dataset ─────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH)
print(f"✅ Loaded: {df_raw.shape}  columns: {list(df_raw.columns)}")

# Chuẩn hoá tên cột (dataset mbti_1.csv có cột 'type' và 'posts')
df_raw.columns = [c.strip().lower() for c in df_raw.columns]
assert "type"  in df_raw.columns, "Missing 'type' column"
assert "posts" in df_raw.columns, "Missing 'posts' column"
df_raw = df_raw.dropna(subset=["type", "posts"]).reset_index(drop=True)


# ── 4.2 Map nhãn MBTI → 4 binary columns ─────────────────────
label_dicts = df_raw["type"].apply(mbti_to_binary)
df_raw = pd.concat([df_raw, pd.DataFrame(list(label_dicts))], axis=1)


# ── 4.3 Preprocessing: truncation + psycholinguistic masking ─

def preprocess_user(raw_posts_str: str) -> str:
    """
    1. Tách bài đăng theo '|||'
    2. Giữ tối đa MAX_POSTS=50 bài
    3. Mỗi bài: cắt MAX_WORDS=70 từ đầu
    4. Mask MBTI keywords → <type>
    5. Nối lại bằng ' ||| '
    """
    posts = raw_posts_str.split("|||")
    processed = []
    for post in posts[:MAX_POSTS]:
        post = post.strip()
        # Cắt tối đa 70 từ
        post = " ".join(post.split()[:MAX_WORDS])
        # Psycholinguistic masking
        post = MASK_REGEX.sub("<type>", post)
        if post:
            processed.append(post)
    return " ||| ".join(processed)

print("⏳ Preprocessing posts (masking + truncation)...")
df_raw["clean_text"] = df_raw["posts"].apply(preprocess_user)
print("✅ Preprocessing done")
print(f"   Sample (first 200 chars):\n   {df_raw['clean_text'].iloc[0][:200]}")


# ── 4.4 Stratified Split (phân tầng trên cả 4 trục) ──────────
# Tạo combo label "0101" để stratify đồng thời 4 trục
df_raw["combo"] = (
    df_raw["label_IE"].astype(str) +
    df_raw["label_SN"].astype(str) +
    df_raw["label_TF"].astype(str) +
    df_raw["label_JP"].astype(str)
)

train_df, temp_df = train_test_split(
    df_raw, test_size=0.40,
    stratify=df_raw["combo"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50,
    stratify=temp_df["combo"], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"\n📊 Split:")
print(f"   Train : {len(train_df)} ({len(train_df)/len(df_raw):.0%})")
print(f"   Val   : {len(val_df)}  ({len(val_df)/len(df_raw):.0%})")
print(f"   Test  : {len(test_df)}  ({len(test_df)/len(df_raw):.0%})")


# ── 4.5 Tính Class Weights từ tập Train ──────────────────────
# Dùng để thiên lệch Few-shot RAG về phía nhãn thiểu số
class_weights = {}
for ax in AXES:
    col = f"label_{ax}"
    y   = train_df[col].values
    cw  = compute_class_weight("balanced", classes=np.array([0, 1]), y=y)
    class_weights[ax] = {"0": float(cw[0]), "1": float(cw[1])}
    minority_cls = "0" if cw[0] > cw[1] else "1"
    print(f"   {ax}: weights={cw.round(3)}  minority={minority_cls}")

print("✅ Class weights ready")

## 🗄️ Cell 5 — Knowledge Base (Feature Extraction + FAISS)

In [ ]:
# ============================================================
# CELL 5 — Xây dựng Knowledge Base
# 5A: Feature Extraction (LLM → Tóm tắt tâm lý)
# 5B: Dual-Embedding → FAISS Index
# ============================================================

import faiss
from tqdm.auto import tqdm
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS as LangFAISS
from langchain.schema import Document

# ─────────────────────────────────────────────────────────────
# 5A. Feature Extraction Agent (Train set)
# [OOM] Xử lý theo batch nhỏ, clear cache sau mỗi N user
# ─────────────────────────────────────────────────────────────

PSYCH_PROMPT_TEMPLATE = """You are a psycholinguistic expert analyzing social media posts.

Analyze the writing style and content of the following posts from ONE person.
Generate a concise psychological profile (4-6 sentences) covering:
1. Social orientation: introversion/extroversion signals
2. Information style: concrete/sensory vs abstract/theoretical language
3. Decision-making: logic/objectivity vs emotion/values references
4. Lifestyle: structured/planned vs spontaneous/flexible signals

Posts:
{posts}

Psychological Profile (output ONLY the profile text, no labels):"""

def extract_psych_summary(posts_text: str, llm) -> str:
    """Gọi LLM để sinh tóm tắt tâm lý cho 1 user."""
    prompt = PSYCH_PROMPT_TEMPLATE.format(
        posts=posts_text[:1200]  # giới hạn context để tránh OOM
    )
    try:
        response = llm.invoke(prompt)
        # Loại bỏ phần lặp lại prompt nếu return_full_text=True
        return str(response).strip()[:600]  # giới hạn output
    except Exception as e:
        print(f"⚠️  LLM error: {e}")
        return posts_text[:300]  # fallback: dùng raw text

# ── Chạy Feature Extraction trên Train set ───────────────────
# [OOM] KB_SAMPLE: set None để dùng toàn bộ, set số nhỏ để test nhanh
# Với 8000 user × ~0.3s/call → ~40 phút (toàn bộ)
# Khuyến nghị: dùng 500-1000 mẫu cho Knowledge Base là đủ
KB_SAMPLE = 800  # Tăng lên None khi chạy đầy đủ

kb_df = train_df.copy()
if KB_SAMPLE and KB_SAMPLE < len(kb_df):
    # Stratified sample để giữ phân phối nhãn
    kb_df = kb_df.groupby("combo", group_keys=False).apply(
        lambda x: x.sample(min(len(x), max(1, int(KB_SAMPLE * len(x) / len(train_df)))),
                           random_state=42)
    ).reset_index(drop=True)
    print(f"📎 Using {len(kb_df)} samples for Knowledge Base (stratified)")

print("⏳ Extracting psychological summaries (LLM)...")
psych_summaries = []
CLEAR_CACHE_EVERY = 50  # [OOM] giải phóng VRAM sau mỗi N user

for i, row in tqdm(kb_df.iterrows(), total=len(kb_df), desc="Feature Extraction"):
    summary = extract_psych_summary(row["clean_text"], local_llm)
    psych_summaries.append(summary)
    if (i + 1) % CLEAR_CACHE_EVERY == 0:
        torch.cuda.empty_cache()  # [OOM] quan trọng

kb_df = kb_df.copy()
kb_df["psych_summary"] = psych_summaries
print(f"✅ Feature extraction done. Sample summary:\n{psych_summaries[0][:200]}")

# [OOM] Clear sau batch lớn
torch.cuda.empty_cache()


# ─────────────────────────────────────────────────────────────
# 5B. Dual-Embedding → FAISS Index
# ─────────────────────────────────────────────────────────────

print("\n⏳ Loading sentence embedder...")
# [OOM] all-MiniLM-L6-v2: 22M params, embed dim 384 → rất nhẹ
# Chạy trên CPU để tiết kiệm VRAM GPU cho LLM
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},  # CPU để tiết kiệm GPU VRAM
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
)

# Tạo LangChain Documents
# Dual embedding: content = raw text + "||" + summary
# Metadata giữ labels và summary riêng để truy vấn sau
documents = []
for _, row in kb_df.iterrows():
    # Nội dung embed: ghép text + summary để capture cả hai khía cạnh
    dual_content = (
        f"[RAW TEXT] {row['clean_text'][:500]}\n\n"
        f"[PSYCHOLOGICAL SUMMARY] {row['psych_summary']}"
    )
    doc = Document(
        page_content=dual_content,
        metadata={
            "label_IE": int(row["label_IE"]),
            "label_SN": int(row["label_SN"]),
            "label_TF": int(row["label_TF"]),
            "label_JP": int(row["label_JP"]),
            "mbti_type": row["type"],
            "psych_summary": row["psych_summary"],
            "raw_text_snippet": row["clean_text"][:300],
        }
    )
    documents.append(doc)

print(f"⏳ Building FAISS index from {len(documents)} documents...")
faiss_store = LangFAISS.from_documents(documents, embedder)
print(f"✅ FAISS index built. Docs: {faiss_store.index.ntotal}")

# Lưu index để reuse (không phải build lại mỗi lần)
faiss_store.save_local(os.path.join(RESULT_DIR, "faiss_kb"))
print(f"   Saved to: {RESULT_DIR}/faiss_kb/")

## 🤝 Cell 6 — Multi-Agent System (6 Agents)

In [ ]:
# ============================================================
# CELL 6 — Hệ thống 6 Agents (RAG + Multi-Agent Pipeline)
# ============================================================

import json
import math
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

# ─────────────────────────────────────────────────────────────
# Data class cho kết quả pipeline
# ─────────────────────────────────────────────────────────────

@dataclass
class AgentResult:
    axis: str
    prediction: int        # 0 hoặc 1
    reasoning: str
    confidence: float = 0.5

@dataclass
class PipelineOutput:
    user_idx: int
    axis_results: Dict[str, AgentResult] = field(default_factory=dict)
    mbti_pred: str = ""
    mbti_true: str = ""
    confidence_score: float = 0.0
    synthesis_reasoning: str = ""


# ─────────────────────────────────────────────────────────────
# AGENT 1 — Feature Extraction Agent
# ─────────────────────────────────────────────────────────────

class FeatureExtractionAgent:
    """Phân tích văn bản test → Tóm tắt tâm lý."""

    PROMPT = """You are a psycholinguistic expert.

Analyze these social media posts from ONE person and write a psychological profile (3-5 sentences):
Focus on: social style, information processing, decision patterns, lifestyle structure.

Posts:
{posts}

Profile (text only, no labels or headers):"""

    def __init__(self, llm):
        self.llm = llm

    def run(self, posts_text: str) -> str:
        prompt = self.PROMPT.format(posts=posts_text[:1200])
        try:
            return str(self.llm.invoke(prompt)).strip()[:500]
        except Exception as e:
            print(f"    [Agent1] Error: {e}")
            return posts_text[:250]


# ─────────────────────────────────────────────────────────────
# AGENT 2 — Dual-RAG Retriever with Balanced Few-Shot
# ─────────────────────────────────────────────────────────────

class DualRAGAgent:
    """
    Truy vấn FAISS với dual embedding.
    Balanced Few-Shot: đảm bảo ít nhất 1 example từ nhãn thiểu số
    cho mỗi trục quan trọng.
    """

    def __init__(self, faiss_store, embedder, class_weights: dict, top_k: int = 5):
        self.store   = faiss_store
        self.embedder= embedder
        self.cw      = class_weights  # {"IE": {"0": w0, "1": w1}, ...}
        self.top_k   = top_k

    def _get_minority_class(self, axis: str) -> int:
        """Trả về nhãn có class weight cao hơn (= nhãn thiểu số)."""
        cw = self.cw[axis]
        return 0 if cw["0"] > cw["1"] else 1

    def run(self, posts_text: str, psych_summary: str) -> List[dict]:
        """
        Retrieve top candidates, sau đó balance theo class weights.
        Return: list of metadata dicts (top-5 examples)
        """
        # Dual query: kết hợp raw text + summary
        query = f"[RAW TEXT] {posts_text[:400]}\n\n[PSYCHOLOGICAL SUMMARY] {psych_summary}"

        # Lấy top_k * 3 để có đủ candidates cho balancing
        k_fetch = min(self.top_k * 3, self.store.index.ntotal)
        raw_results = self.store.similarity_search(query, k=k_fetch)

        candidates = [doc.metadata for doc in raw_results]
        if not candidates:
            return []

        # ── Balanced selection ────────────────────────────────
        # Với mỗi trục, đảm bảo có ít nhất 1 example minority
        selected_idxs = set()
        priority_idxs = []

        for axis in AXES:
            minority = self._get_minority_class(axis)
            col      = f"label_{axis}"
            for j, meta in enumerate(candidates):
                if meta.get(col) == minority and j not in selected_idxs:
                    priority_idxs.append(j)
                    selected_idxs.add(j)
                    break  # 1 minority per axis đã đủ

        # Điền phần còn lại từ top candidates
        for j in range(len(candidates)):
            if len(selected_idxs) >= self.top_k:
                break
            selected_idxs.add(j)

        final_idxs = sorted(selected_idxs)[:self.top_k]
        return [candidates[j] for j in final_idxs]


# ─────────────────────────────────────────────────────────────
# AGENT 3 — Analysis Agent (1 per axis)
# ─────────────────────────────────────────────────────────────

class AnalysisAgent:
    """
    Phân tích 1 trục MBTI dựa trên:
    - Văn bản gốc + Tóm tắt tâm lý
    - Top-5 RAG examples (few-shot)
    - Label Enrichment của trục đó
    Ép LLM trả về JSON: {"prediction": 0|1, "reasoning": "..."}
    """

    PROMPT = """You are an expert MBTI analyst specializing in the {axis} dimension.

=== AXIS KNOWLEDGE ===
{axis_knowledge}

=== USER POSTS ===
{posts}

=== PSYCHOLOGICAL SUMMARY ===
{summary}

=== SIMILAR EXAMPLES (Few-Shot) ===
{examples}

=== TASK ===
Based on ALL evidence above, predict whether this person is {label_0} (0) or {label_1} (1) on the {axis} axis.

Respond with ONLY a valid JSON object, no extra text:
{{"prediction": <0 or 1>, "reasoning": "<one concise sentence explaining key signals>"}}"""

    def __init__(self, axis: str, llm):
        self.axis = axis
        self.llm  = llm
        self.label_0, self.label_1 = AXIS_LABELS[axis]

    def _format_examples(self, rag_docs: List[dict]) -> str:
        if not rag_docs:
            return "No examples available."
        lines = []
        for i, doc in enumerate(rag_docs, 1):
            ax_val  = doc.get(f"label_{self.axis}", "?")
            mbti    = doc.get("mbti_type", "?")
            snippet = doc.get("raw_text_snippet", "")[:120]
            summ    = doc.get("psych_summary", "")[:100]
            lines.append(
                f"Example {i}: MBTI={mbti} | {self.axis}={ax_val}\n"
                f"  Text: {snippet}...\n"
                f"  Profile: {summ}..."
            )
        return "\n".join(lines)

    def _parse_json_response(self, raw: str) -> Tuple[int, str]:
        """Robust JSON parsing với fallback."""
        raw = raw.strip()
        # Thử tìm JSON block trong response
        json_match = re.search(r"\{[^{}]*\"prediction\"[^{}]*\}", raw, re.DOTALL)
        if json_match:
            try:
                parsed = json.loads(json_match.group())
                pred = int(parsed.get("prediction", 0))
                pred = max(0, min(1, pred))  # clamp to 0/1
                reasoning = str(parsed.get("reasoning", ""))
                return pred, reasoning
            except Exception:
                pass
        # Fallback: tìm số 0 hoặc 1 trong text
        numbers = re.findall(r"\b([01])\b", raw)
        if numbers:
            return int(numbers[0]), raw[:100]
        return 0, f"Parse failed. Raw: {raw[:80]}"

    def run(self, posts_text: str, psych_summary: str,
            rag_docs: List[dict]) -> AgentResult:
        prompt = self.PROMPT.format(
            axis          = self.axis,
            axis_knowledge= LABEL_ENRICHMENT[self.axis],
            posts         = posts_text[:600],
            summary       = psych_summary[:300],
            examples      = self._format_examples(rag_docs),
            label_0       = self.label_0,
            label_1       = self.label_1,
        )
        try:
            raw_resp = str(self.llm.invoke(prompt)).strip()
            pred, reasoning = self._parse_json_response(raw_resp)
        except Exception as e:
            print(f"    [Agent3-{self.axis}] Error: {e}")
            pred, reasoning = 0, "Error in LLM call"

        return AgentResult(
            axis       = self.axis,
            prediction = pred,
            reasoning  = reasoning,
        )


# ─────────────────────────────────────────────────────────────
# AGENT 4 — Synthesis Agent
# ─────────────────────────────────────────────────────────────

class SynthesisAgent:
    """
    Tổng hợp 4 quyết định trục → MBTI type + confidence.
    Confidence = trung bình entropy nhị phân nghịch đảo:
      H(p) = -p*log2(p) - (1-p)*log2(1-p)
      confidence = 1 - mean(H) / log2(2)   [normalize 0→1]
    Nếu model trả về soft probability, dùng đó; không thì
    dùng heuristic từ reasoning text.
    """

    PROMPT = """You are a senior MBTI psychologist. Synthesize the 4 axis predictions below.

Predictions:
{axis_predictions}

Task: Determine the final MBTI type and overall confidence (0.0-1.0).

Respond ONLY with valid JSON:
{{"mbti": "<4-letter type>", "confidence": <0.0-1.0>, "reasoning": "<brief synthesis>"}}"""

    def __init__(self, llm):
        self.llm = llm

    def _binary_entropy(self, p: float) -> float:
        p = max(1e-9, min(1 - 1e-9, p))
        return -p * math.log2(p) - (1 - p) * math.log2(1 - p)

    def _compute_confidence(self, axis_results: Dict[str, AgentResult]) -> float:
        """
        Tính confidence từ mức độ chắc chắn của 4 quyết định.
        Proxy: keywords trong reasoning gợi ý confidence cao/thấp.
        """
        high_conf_words = {"clearly", "strongly", "definitely", "obvious",
                           "consistently", "very", "highly"}
        low_conf_words  = {"maybe", "possibly", "uncertain", "slight",
                           "borderline", "unclear", "mixed"}
        scores = []
        for ar in axis_results.values():
            text = ar.reasoning.lower()
            h = sum(1 for w in high_conf_words if w in text)
            l = sum(1 for w in low_conf_words  if w in text)
            # Map heuristic → probability-like score
            raw_conf = 0.5 + 0.1 * h - 0.1 * l
            raw_conf = max(0.5, min(0.95, raw_conf))
            # Low entropy = high confidence
            scores.append(1.0 - self._binary_entropy(raw_conf))
        return round(float(np.mean(scores)), 3)

    def _parse_synthesis(self, raw: str, fallback_mbti: str,
                         fallback_conf: float) -> Tuple[str, float, str]:
        json_match = re.search(r"\{[^{}]*\"mbti\"[^{}]*\}", raw, re.DOTALL)
        if json_match:
            try:
                parsed = json.loads(json_match.group())
                mbti  = str(parsed.get("mbti", fallback_mbti)).upper().strip()
                # Validasi format MBTI
                if len(mbti) == 4 and all(c in "IESNTFJP" for c in mbti):
                    conf = float(parsed.get("confidence", fallback_conf))
                    conf = max(0.0, min(1.0, conf))
                    reasoning = str(parsed.get("reasoning", ""))
                    return mbti, conf, reasoning
            except Exception:
                pass
        return fallback_mbti, fallback_conf, raw[:100]

    def run(self, axis_results: Dict[str, AgentResult],
            user_idx: int) -> Tuple[str, float, str]:
        # Fallback MBTI từ individual predictions
        ie = axis_results.get("IE", AgentResult("IE", 0, "")).prediction
        sn = axis_results.get("SN", AgentResult("SN", 1, "")).prediction
        tf = axis_results.get("TF", AgentResult("TF", 0, "")).prediction
        jp = axis_results.get("JP", AgentResult("JP", 0, "")).prediction
        fallback_mbti = labels_to_mbti(ie, sn, tf, jp)
        fallback_conf = self._compute_confidence(axis_results)

        # Gọi LLM synthesis
        axis_pred_str = "\n".join([
            f"  {ax}: prediction={ar.prediction} "
            f"({'I/S/T/J'[list(AXES).index(ax)] if ar.prediction==0 else 'E/N/F/P'[list(AXES).index(ax)]})"
            f" | reasoning: {ar.reasoning[:100]}"
            for ax, ar in axis_results.items()
        ])

        try:
            raw = str(self.llm.invoke(
                self.PROMPT.format(axis_predictions=axis_pred_str)
            )).strip()
            mbti, conf, reasoning = self._parse_synthesis(
                raw, fallback_mbti, fallback_conf
            )
        except Exception as e:
            print(f"    [Synthesis] Error: {e}")
            mbti, conf, reasoning = fallback_mbti, fallback_conf, "LLM error"

        return mbti, conf, reasoning


# ─────────────────────────────────────────────────────────────
# Khởi tạo tất cả Agents
# ─────────────────────────────────────────────────────────────

agent_feature   = FeatureExtractionAgent(llm=local_llm)
agent_rag       = DualRAGAgent(
    faiss_store    = faiss_store,
    embedder       = embedder,
    class_weights  = class_weights,
    top_k          = 5,
)
analysis_agents = {ax: AnalysisAgent(axis=ax, llm=local_llm) for ax in AXES}
agent_synthesis = SynthesisAgent(llm=local_llm)

print("✅ All 6 agents initialized:")
print("   Agent 1: Feature Extraction")
print("   Agent 2: Dual-RAG (Balanced Few-Shot)")
print("   Agent 3: Analysis × 4 (IE, SN, TF, JP)")
print("   Agent 4: Synthesis")

## 🚀 Cell 7 — Inference + Evaluation + Save CSV

In [ ]:
# ============================================================
# CELL 7 — Full Pipeline: Inference + Evaluation + Save CSV
# ============================================================

import time
import concurrent.futures
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report
)

# ─────────────────────────────────────────────────────────────
# 7.1 — Nhận diện phần cứng & tối ưu luồng xử lý
# ─────────────────────────────────────────────────────────────

NUM_GPUS = torch.cuda.device_count()

def get_num_workers() -> int:
    """
    Với LLM inference tuần tự (1 request/time do GPU share),
    dùng threading để overlap I/O (FAISS search, text processing)
    với LLM compute.
    1 GPU → 1 worker (LLM là bottleneck)
    2 GPU → có thể thử 2, nhưng LLM.invoke không thread-safe với
             shared model → vẫn dùng 1 worker, chỉ parallel FAISS
    """
    return 1

PIPELINE_WORKERS = get_num_workers()
print(f"🖥  Hardware: {NUM_GPUS} GPU(s) | Pipeline workers: {PIPELINE_WORKERS}")
if NUM_GPUS == 2:
    print("   Note: 2 GPU detected. DataParallel active for LLM forward pass.")
    print("   Analysis agents run sequentially to avoid race conditions.")


# ─────────────────────────────────────────────────────────────
# 7.2 — Single-user pipeline
# ─────────────────────────────────────────────────────────────

def run_pipeline_for_user(row_data: dict, user_idx: int) -> PipelineOutput:
    """
    Chạy toàn bộ pipeline RAG + Multi-Agent cho 1 user.
    Args:
        row_data: dict với keys 'clean_text', 'type', label_* cols
        user_idx: integer index
    Returns: PipelineOutput
    """
    output = PipelineOutput(user_idx=user_idx)
    posts_text = row_data["clean_text"]

    # ── Agent 1: Feature Extraction ──────────────────────────
    psych_summary = agent_feature.run(posts_text)
    torch.cuda.empty_cache()  # [OOM] sau mỗi LLM call

    # ── Agent 2: Dual-RAG Retrieval ──────────────────────────
    rag_docs = agent_rag.run(posts_text, psych_summary)

    # ── Agent 3: 4 Analysis Agents (tuần tự vì shared GPU) ──
    for ax in AXES:
        result = analysis_agents[ax].run(posts_text, psych_summary, rag_docs)
        output.axis_results[ax] = result
        torch.cuda.empty_cache()  # [OOM] sau mỗi axis

    # ── Agent 4: Synthesis ───────────────────────────────────
    mbti_pred, conf, reasoning = agent_synthesis.run(
        output.axis_results, user_idx
    )
    torch.cuda.empty_cache()

    output.mbti_pred            = mbti_pred
    output.mbti_true            = row_data.get("type", "").upper()
    output.confidence_score     = conf
    output.synthesis_reasoning  = reasoning

    return output


# ─────────────────────────────────────────────────────────────
# 7.3 — Batch inference trên Test set
# ─────────────────────────────────────────────────────────────

# [Tuỳ chỉnh] TEST_SAMPLE = None để chạy toàn bộ test set
# Với T4 ~0.3-0.8s/user × 5 agents → ~2-4s/user
# 1700 user × 4s ≈ ~2 giờ. Dùng sample nhỏ để demo nhanh.
TEST_SAMPLE = 200  # Set None cho full evaluation

eval_df = test_df.copy()
if TEST_SAMPLE and TEST_SAMPLE < len(eval_df):
    eval_df = eval_df.sample(n=TEST_SAMPLE, random_state=42).reset_index(drop=True)
    print(f"📎 Evaluating on {len(eval_df)} test samples (sampled)")
else:
    print(f"📎 Evaluating on full test set: {len(eval_df)} samples")

results_list = []
start_time   = time.time()

for idx in tqdm(range(len(eval_df)), desc="Pipeline Inference"):
    row = eval_df.iloc[idx].to_dict()
    try:
        out = run_pipeline_for_user(row, user_idx=idx)
        results_list.append(out)
    except Exception as e:
        print(f"\n⚠️  User {idx} failed: {e}")
        # Append dummy output để không mất index
        dummy = PipelineOutput(user_idx=idx,
                               mbti_pred="INTJ",
                               mbti_true=row.get("type","INTJ").upper(),
                               confidence_score=0.0)
        for ax in AXES:
            dummy.axis_results[ax] = AgentResult(ax, 0, "pipeline_error")
        results_list.append(dummy)

elapsed = time.time() - start_time
print(f"\n⏱  Inference done: {elapsed:.1f}s total | {elapsed/len(eval_df):.1f}s/user")


# ─────────────────────────────────────────────────────────────
# 7.4 — Build results DataFrame & Compute Metrics
# ─────────────────────────────────────────────────────────────

rows = []
for i, out in enumerate(results_list):
    true_row = eval_df.iloc[i]
    row_dict = {
        "user_index":    out.user_idx,
        # Ground truth
        "y_true_IE":     int(true_row["label_IE"]),
        "y_true_SN":     int(true_row["label_SN"]),
        "y_true_TF":     int(true_row["label_TF"]),
        "y_true_JP":     int(true_row["label_JP"]),
        "y_true_16type": out.mbti_true,
        # Predictions
        "y_pred_IE":     int(out.axis_results.get("IE", AgentResult("IE",0,"")).prediction),
        "y_pred_SN":     int(out.axis_results.get("SN", AgentResult("SN",0,"")).prediction),
        "y_pred_TF":     int(out.axis_results.get("TF", AgentResult("TF",0,"")).prediction),
        "y_pred_JP":     int(out.axis_results.get("JP", AgentResult("JP",0,"")).prediction),
        "y_pred_16type": out.mbti_pred,
        "confidence_score": out.confidence_score,
        # Extra diagnostics
        "reasoning_IE":  out.axis_results.get("IE", AgentResult("IE",0,"")).reasoning[:100],
        "reasoning_SN":  out.axis_results.get("SN", AgentResult("SN",0,"")).reasoning[:100],
        "reasoning_TF":  out.axis_results.get("TF", AgentResult("TF",0,"")).reasoning[:100],
        "reasoning_JP":  out.axis_results.get("JP", AgentResult("JP",0,"")).reasoning[:100],
        "synthesis_reasoning": out.synthesis_reasoning[:150],
    }
    rows.append(row_dict)

results_df = pd.DataFrame(rows)

# ── Metrics ──────────────────────────────────────────────────
print("\n" + "="*60)
print("📊  EVALUATION RESULTS — RAG + Multi-Agent System")
print("="*60)

axis_metrics = {}
for ax, dim_name in zip(AXES, DIM_NAMES):
    y_true = results_df[f"y_true_{ax}"].values
    y_pred = results_df[f"y_pred_{ax}"].values
    acc    = accuracy_score(y_true, y_pred)
    f1     = f1_score(y_true, y_pred, average="macro", zero_division=0)
    axis_metrics[ax] = {"accuracy": acc, "macro_f1": f1}
    print(f"  {dim_name:5s} | Acc={acc:.4f} | Macro-F1={f1:.4f}")

avg_f1  = np.mean([m["macro_f1"]  for m in axis_metrics.values()])
avg_acc = np.mean([m["accuracy"]  for m in axis_metrics.values()])

# Exact Match (16 nhãn)
exact_match = (results_df["y_true_16type"] == results_df["y_pred_16type"]).mean()

print(f"\n  {'Average':5s} | Acc={avg_acc:.4f} | Macro-F1={avg_f1:.4f}")
print(f"  Exact Match (16-type): {exact_match:.4f} ({exact_match*100:.1f}%)")
print("="*60)

# Phân phối confidence
print(f"\n  Confidence score stats:")
print(f"  mean={results_df['confidence_score'].mean():.3f} | "
      f"min={results_df['confidence_score'].min():.3f} | "
      f"max={results_df['confidence_score'].max():.3f}")


# ─────────────────────────────────────────────────────────────
# 7.5 — Lưu CSV (BẮT BUỘC)
# ─────────────────────────────────────────────────────────────

# File 1: Core predictions (theo spec)
core_cols = [
    "user_index",
    "y_true_IE", "y_pred_IE",
    "y_true_SN", "y_pred_SN",
    "y_true_TF", "y_pred_TF",
    "y_true_JP", "y_pred_JP",
    "y_true_16type", "y_pred_16type",
    "confidence_score",
]
save_path = os.path.join(RESULT_DIR, "rag_multi_agent_predictions.csv")
results_df[core_cols].to_csv(save_path, index=False)
print(f"\n✅ Saved: {save_path}")

# File 2: Full results với reasoning (diagnostics)
full_path = os.path.join(RESULT_DIR, "rag_multi_agent_predictions_full.csv")
results_df.to_csv(full_path, index=False)
print(f"✅ Saved: {full_path}")

# File 3: Summary metrics JSON
metrics_summary = {
    "eval_samples": len(results_df),
    "elapsed_seconds": round(elapsed, 1),
    "avg_macro_f1": round(avg_f1, 4),
    "avg_accuracy": round(avg_acc, 4),
    "exact_match_16type": round(exact_match, 4),
    "per_axis": {
        ax: {k: round(v, 4) for k, v in m.items()}
        for ax, m in axis_metrics.items()
    }
}
with open(os.path.join(RESULT_DIR, "metrics_summary.json"), "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"✅ Saved: {RESULT_DIR}/metrics_summary.json")

print("\n📁 All results in:", os.path.abspath(RESULT_DIR))
print("   Files:", os.listdir(RESULT_DIR))

## 📊 Cell 8 — Final Comparison Table

In [ ]:
# ============================================================
# CELL 8 — Bảng tổng hợp & Phân tích kết quả
# ============================================================

# ── 8.1 Đọc lại CSV và in bảng đẹp ──────────────────────────
pred_df = pd.read_csv(os.path.join(RESULT_DIR, "rag_multi_agent_predictions.csv"))

print("="*65)
print("📊  KẾT QUẢ CUỐI CÙNG — RAG + Multi-Agent (No Fine-tuning)")
print(f"    Model: {MODEL_ID}")
print(f"    Dataset: {DATA_PATH}  |  Test samples: {len(pred_df)}")
print("="*65)

summary_rows = []
for ax, dim_name in zip(AXES, DIM_NAMES):
    yt = pred_df[f"y_true_{ax}"]
    yp = pred_df[f"y_pred_{ax}"]
    summary_rows.append({
        "Axis":      dim_name,
        "Accuracy":  round(accuracy_score(yt, yp), 4),
        "Macro-F1":  round(f1_score(yt, yp, average="macro", zero_division=0), 4),
        "F1-class0": round(f1_score(yt, yp, pos_label=0, average="binary", zero_division=0), 4),
        "F1-class1": round(f1_score(yt, yp, pos_label=1, average="binary", zero_division=0), 4),
    })

summary_df = pd.DataFrame(summary_rows)

# Row tổng hợp
avg_row = {
    "Axis":      "AVERAGE",
    "Accuracy":  round(summary_df["Accuracy"].mean(), 4),
    "Macro-F1":  round(summary_df["Macro-F1"].mean(), 4),
    "F1-class0": round(summary_df["F1-class0"].mean(), 4),
    "F1-class1": round(summary_df["F1-class1"].mean(), 4),
}
exact_m = round((pred_df["y_true_16type"] == pred_df["y_pred_16type"]).mean(), 4)

print(summary_df.to_string(index=False))
print("-"*65)
print(f"  AVERAGE   | Acc={avg_row['Accuracy']} | Macro-F1={avg_row['Macro-F1']}")
print(f"  Exact Match (16-class): {exact_m}  ({exact_m*100:.1f}%)")
print("="*65)

# ── 8.2 Phân phối dự đoán ────────────────────────────────────
print("\n📊 Prediction distribution (y_pred_16type):")
pred_dist = pred_df["y_pred_16type"].value_counts()
print(pred_dist.to_string())

print("\n📊 Ground truth distribution (y_true_16type):")
true_dist = pred_df["y_true_16type"].value_counts()
print(true_dist.to_string())

# ── 8.3 Confidence phân tích ─────────────────────────────────
high_conf = pred_df[pred_df["confidence_score"] >= 0.7]
if len(high_conf) > 0:
    hc_exact = (high_conf["y_true_16type"] == high_conf["y_pred_16type"]).mean()
    print(f"\n📊 High-confidence predictions (≥0.7):")
    print(f"   Count: {len(high_conf)}/{len(pred_df)} ({len(high_conf)/len(pred_df)*100:.1f}%)")
    print(f"   Exact match among high-conf: {hc_exact:.4f}")

# ── 8.4 Lưu summary table ────────────────────────────────────
summary_df.to_csv(os.path.join(RESULT_DIR, "metrics_per_axis.csv"), index=False)
print(f"\n✅ Per-axis metrics saved → {RESULT_DIR}/metrics_per_axis.csv")

# ── 8.5 System info ──────────────────────────────────────────
print("\n🖥  System Info:")
print(f"   GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {alloc:.2f}/{total:.2f} GB VRAM used")
print(f"   LLM: {MODEL_ID} (4-bit quantized, no fine-tuning)")
print(f"   Embedder: all-MiniLM-L6-v2 (CPU)")
print(f"   KB size: {faiss_store.index.ntotal} documents")
print(f"\n✅ Pipeline complete!")